# 3b단계: 팀 분류기 재학습 (PyTorch ResNet18)

3a에서 확인한 문제: v3 RF 분류기가 **홍익 쪽으로 치우쳐** 국민대 1/3과 심판 전원을 홍익으로 분류.
이 노트북은 **튜닝 클립에서 직접 라벨한 트랙**으로 CNN 분류기를 새로 학습하고, 평가 클립에서 v4와 비교합니다.
탐지·추적은 2a·2b 캐시를 그대로 써요 (탐지 다시 안 함).

**실행 순서:** 위에서부터 순서대로 → ③에서 버튼으로 라벨링 → 나머지 끝까지

**런타임: T4 GPU** · 전체 약 30~40분 (라벨링 포함)

## 공통 설정

In [ ]:
import os, json, glob, shutil, zipfile, re
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from scipy.optimize import linear_sum_assignment
from google.colab import drive
drive.mount('/content/drive')

# 한글 글꼴 (그래프의 한글이 네모로 깨지지 않게)
import subprocess, matplotlib.font_manager as fm
_font = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
if not os.path.exists(_font):
    subprocess.run("apt-get -qq install -y fonts-nanum > /dev/null", shell=True)
if os.path.exists(_font):
    fm.fontManager.addfont(_font)
    plt.rcParams["font.family"] = "NanumGothic"
else:
    print("⚠️ 한글 글꼴 설치 실패: 그래프의 한글이 네모로 보일 수 있어요")
plt.rcParams["axes.unicode_minus"] = False

In [ ]:
# ===== 경로 (v3 노트북 Cell 2와 동일하게) =====
FOLDER_PATH      = "/content/drive/MyDrive/2025년 홍익 vs 국민 드론뷰"
DRIVE_OUTPUT_DIR = os.path.join(FOLDER_PATH, "pipeline_outputs")   # v3의 tracking_final.json 위치
EVAL_DIR         = os.path.join(DRIVE_OUTPUT_DIR, "eval")           # 이번에 새로 만드는 평가 폴더
IMG_DIR          = os.path.join(EVAL_DIR, "images")                 # 평가용 프레임 20장 (정답 라벨의 기준 이미지)
PRE_DIR          = os.path.join(EVAL_DIR, "prelabels")              # 사전 라벨
for _d in (EVAL_DIR, IMG_DIR, PRE_DIR):
    os.makedirs(_d, exist_ok=True)

# ===== 클래스 =====
# 정답(GT)에는 0~2만 남아야 합니다. 3(Unknown)은 "라벨러가 확인해야 할 박스" 표시용입니다.
CLASS_NAMES = ["Hongik", "Kookmin", "Referee", "Unknown"]
CLS_ID = {n: i for i, n in enumerate(CLASS_NAMES)}

# ===== 평가 클립 =====
# play_range : 득점 플레이 구간 (v3 frame_id 기준, 1부터 시작, 양 끝 포함)
# scorer     : v3에서 득점 선수 경로를 만들 때 쓴 (track_id, 프레임 구간). 득점 선수 '정답 위치'로 사용.
#              ⚠️ 0156의 #171, #183 구간은 v3 노트북 주석에 '추정, 확인 필요'로 되어 있었습니다.
#                 v3의 프레임 브라우저 셀로 확인 후 틀렸다면 여기 숫자를 고치세요.
EVAL_CLIPS = {
    "DJI_20250913133612_0144_D": {
        "clip_key": 34,
        "play_range": (413, 649),
        "scorer": [(2122, (413, 649))],
    },
    "DJI_20250913135020_0156_D": {
        "clip_key": 44,
        "play_range": (167, 525),
        "scorer": [(184, (167, 214)), (171, (215, 313)), (183, (314, 393)), (208, (410, 525))],
    },
}
FRAMES_PER_CLIP = 10

# ===== 매칭 기준 =====
# 선수가 10~25px로 작아서 IoU 0.5는 박스가 조금만 어긋나도 오답이 됩니다.
# 대신 '박스 중심 거리'로 짝을 짓습니다: 거리 <= max(MIN_PX, RATIO × 정답 박스 긴 변)
MATCH_MIN_PX = 10
MATCH_RATIO  = 0.5
SCORER_MATCH_PX = 15   # 득점 선수 정답 위치(발)와 예측 박스 발 위치가 이 거리(px) 이내면 '찾았다'


def video_path_of(stem):
    for ext in (".MP4", ".mp4", ".MOV", ".mov"):
        p = os.path.join(FOLDER_PATH, stem + ext)
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f"영상을 찾을 수 없습니다: {stem}")

def tracking_json_of(stem, folder=DRIVE_OUTPUT_DIR):
    return os.path.join(folder, f"{stem}_tracking_final.json")

for stem in EVAL_CLIPS:
    print(stem, "| 영상:", os.path.exists(video_path_of(stem)), "| v3 JSON:", os.path.exists(tracking_json_of(stem)))

In [ ]:
# ===== 공통 함수 =====

def load_tracking(json_path):
    """v3 형식 JSON → {frame_id: [player dict, ...]}, field_polygon"""
    with open(json_path, encoding="utf-8") as f:
        data = json.load(f)
    by_frame = {fr["frame_id"]: fr["players"] for fr in data["frames"]}
    poly = np.array(data.get("field_polygon", []), dtype=np.int32)
    return by_frame, poly


def read_frames_sequential(video_path, frame_ids):
    """
    frame_id(1부터 시작)에 해당하는 프레임을 '처음부터 순서대로 읽어서' 가져옵니다.
    cap.set(POS_FRAMES)로 바로 점프하면 드론 영상(H.264/H.265)에서 몇 프레임씩 어긋날 수 있어서,
    v3가 프레임을 센 방식(처음부터 1, 2, 3...)과 똑같이 맞추기 위해 순차로 읽습니다.
    """
    wanted = set(frame_ids)
    last = max(wanted)
    out = {}
    cap = cv2.VideoCapture(video_path)
    fid = 0
    while fid < last:
        ok, frame = cap.read()
        if not ok:
            break
        fid += 1
        if fid in wanted:
            out[fid] = frame
    cap.release()
    missing = wanted - set(out)
    if missing:
        print(f"⚠️ 읽지 못한 프레임: {sorted(missing)}")
    return out


def pick_frames(play_range, n):
    a, b = play_range
    return sorted(set(np.linspace(a, b, n).round().astype(int).tolist()))


def inside(poly, x, y):
    if poly is None or len(poly) < 3:
        return True
    return cv2.pointPolygonTest(poly.reshape(-1, 1, 2), (float(x), float(y)), False) >= 0


def xyxy_to_yolo_line(cls_id, box, W, H):
    x1, y1, x2, y2 = box
    x1, x2 = max(0, x1), min(W, x2)
    y1, y2 = max(0, y1), min(H, y2)
    cx, cy = (x1 + x2) / 2 / W, (y1 + y2) / 2 / H
    w, h = (x2 - x1) / W, (y2 - y1) / H
    return f"{cls_id} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}"


def yolo_line_to_xyxy(line, W, H):
    c, cx, cy, w, h = line.split()[:5]
    cx, cy, w, h = float(cx) * W, float(cy) * H, float(w) * W, float(h) * H
    return int(c), [cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2]


def center(b):
    return ((b[0] + b[2]) / 2, (b[1] + b[3]) / 2)


def box_iou(a, b):
    """Day 2에서 만든 IoU (xyxy)"""
    iw = max(0.0, min(a[2], b[2]) - max(a[0], b[0]))
    ih = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = iw * ih
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / union if union > 0 else 0.0


def match_by_center(gt_boxes, pred_boxes):
    """
    정답 박스와 예측 박스를 1:1로 짝짓기 (헝가리안 알고리즘, 중심 거리 기준).
    return: [(gi, pi), ...], 짝 없는 정답 인덱스, 짝 없는 예측 인덱스
    """
    G, P = len(gt_boxes), len(pred_boxes)
    if G == 0 or P == 0:
        return [], list(range(G)), list(range(P))
    gc = np.array([center(b) for b in gt_boxes])
    pc = np.array([center(b) for b in pred_boxes])
    dist = np.linalg.norm(gc[:, None, :] - pc[None, :, :], axis=2)          # (G, P)
    gsize = np.array([max(b[2] - b[0], b[3] - b[1]) for b in gt_boxes])
    thr = np.maximum(MATCH_MIN_PX, MATCH_RATIO * gsize)[:, None]            # 정답 박스별 허용 거리
    cost = np.where(dist <= thr, dist, 1e6)
    gi, pi = linear_sum_assignment(cost)
    pairs = [(g, p) for g, p in zip(gi, pi) if cost[g, p] < 1e6]
    mg = {g for g, _ in pairs}
    mp = {p for _, p in pairs}
    return pairs, [g for g in range(G) if g not in mg], [p for p in range(P) if p not in mp]


print("✅ 공통 함수 준비 완료")

## 평가 준비 (0단계 파트 B와 같은 코드)

In [ ]:
GT_ZIP = os.path.join(EVAL_DIR, "gt_roboflow.zip")
GT_DIR = os.path.join("/content", "gt_extracted")
shutil.rmtree(GT_DIR, ignore_errors=True)
with zipfile.ZipFile(GT_ZIP) as z:
    z.extractall(GT_DIR)


def original_name(fname):
    """Roboflow가 바꾼 파일명 → 원래 이름. 예: 'DJI_..._f00413_jpg.rf.ab12cd.txt' → 'DJI_..._f00413'"""
    base = os.path.basename(fname)
    base = re.split(r"_(?:jpg|jpeg|png)\.rf\.", base, flags=re.IGNORECASE)[0]
    return os.path.splitext(base)[0]


def read_names_yaml(gt_dir):
    """Roboflow data.yaml의 클래스 순서를 읽음 (Roboflow가 순서를 바꿀 수 있어서 이름으로 다시 매핑)"""
    ymls = glob.glob(os.path.join(gt_dir, "**", "data.yaml"), recursive=True)
    if not ymls:
        return CLASS_NAMES
    txt = open(ymls[0], encoding="utf-8").read()
    m = re.search(r"names:\s*\[(.*?)\]", txt, re.S)
    if m:
        return [s.strip().strip("'\"") for s in m.group(1).split(",") if s.strip()]
    return re.findall(r"^\s*-\s*['\"]?([^'\"\n]+?)['\"]?\s*$", txt.split("names:")[1], re.M)


def load_gt(gt_dir):
    """→ {name: {"W","H","boxes":[(class_name, xyxy), ...]}}"""
    rf_names = read_names_yaml(gt_dir)
    gt = {}
    for lbl in glob.glob(os.path.join(gt_dir, "**", "labels", "*.txt"), recursive=True):
        name = original_name(lbl)
        img = cv2.imread(os.path.join(IMG_DIR, name + ".jpg"))   # 원본 해상도 기준으로 복원
        if img is None:
            print("⚠️ 원본 이미지를 찾을 수 없음:", name); continue
        H, W = img.shape[:2]
        boxes = []
        for line in open(lbl).read().strip().splitlines():
            if not line.strip():
                continue
            c, b = yolo_line_to_xyxy(line, W, H)
            boxes.append((rf_names[c], b))
        gt[name] = {"W": W, "H": H, "boxes": boxes}
    return gt


GT = load_gt(GT_DIR)
_cnt = Counter(c for v in GT.values() for c, _ in v["boxes"])
print(f"정답 이미지 {len(GT)}장, 박스 {sum(_cnt.values())}개 → {dict(_cnt)}")
if _cnt.get("Unknown"):
    print(f"⚠️ 정답에 Unknown이 {_cnt['Unknown']}개 남아 있습니다. 탐지 평가에는 포함되지만 팀 정확도에서는 제외됩니다.")

In [ ]:
EVAL_TEAM = ["Hongik", "Kookmin", "Referee"]


def detection_and_team_metrics(run_json_by_stem):
    """정답 vs 예측: 탐지(클래스 무관) P/R/F1 + 짝지어진 박스의 클래스 정확도 + 혼동행렬"""
    tp = fp = fn = 0
    conf_mat = Counter()                     # (정답 클래스, 예측 클래스)
    per_frame = []
    for stem, jpath in run_json_by_stem.items():
        by_frame, _ = load_tracking(jpath)
        for name, g in GT.items():
            if not name.startswith(stem):
                continue
            fid = int(name.split("_f")[-1])
            preds = by_frame.get(fid, [])
            gb = [b for _, b in g["boxes"]]
            pb = [p["bbox"] for p in preds]
            pairs, ug, up = match_by_center(gb, pb)
            tp += len(pairs); fn += len(ug); fp += len(up)
            for gi, pi in pairs:
                conf_mat[(g["boxes"][gi][0], preds[pi]["type"])] += 1
            per_frame.append({"image": name, "gt": len(gb), "pred": len(pb), "TP": len(pairs), "FN": len(ug), "FP": len(up)})
    P = tp / (tp + fp) if tp + fp else 0.0
    R = tp / (tp + fn) if tp + fn else 0.0
    F1 = 2 * P * R / (P + R) if P + R else 0.0
    team_pairs = {k: v for k, v in conf_mat.items() if k[0] in EVAL_TEAM}
    correct = sum(v for (g, p), v in team_pairs.items() if g == p)
    team_acc = correct / sum(team_pairs.values()) if team_pairs else 0.0
    ref_gt = sum(1 for v in GT.values() for c, _ in v["boxes"] if c == "Referee")
    ref_hit = conf_mat.get(("Referee", "Referee"), 0)
    return {
        "det_precision": round(P, 3), "det_recall": round(R, 3), "det_f1": round(F1, 3),
        "TP": tp, "FP": fp, "FN": fn,
        "team_acc": round(team_acc, 3),
        "referee_recall": round(ref_hit / ref_gt, 3) if ref_gt else None,
    }, conf_mat, pd.DataFrame(per_frame)


def scorer_track_metrics(stem, run_json, cfg, verbose=True):
    """
    득점 선수 추적 품질 (라벨링 없이 측정).
    - 정답 위치: v3에서 사람이 확인해 이어붙인 득점 선수의 발 위치 (cfg["scorer"])
    - 평가할 결과(run_json)에서 매 프레임 그 위치에 가장 가까운 박스의 track_id를 읽음

    지표
    - scorer_ids / scorer_id_switches : '정답 위치가 확실한 프레임'에서만 계산 (믿을 수 있는 값)
    - scorer_coverage : v3가 놓친 구간은 앞뒤 위치를 직선으로 이어 정답 위치를 '추정'해서 포함
                        → 그 구간에서는 근처의 다른 선수가 잡힐 수 있으니 참고용(근사값)
    """
    ref_by_frame, _ = load_tracking(tracking_json_of(stem))       # 정답 경로는 항상 v3 기준
    run_by_frame, _ = load_tracking(run_json)
    known = {}
    for tid, (a, b) in cfg["scorer"]:
        for fid in range(a, b + 1):
            for p in ref_by_frame.get(fid, []):
                if p["track_id"] == tid:
                    known[fid] = p["feet"]
    if not known:
        print(f"⚠️ {stem}: v3 JSON에서 득점 선수 track_id를 찾지 못했습니다. EVAL_CLIPS의 scorer 값을 확인하세요.")
        return {"scorer_frames": 0, "scorer_coverage": 0.0, "scorer_ids": 0, "scorer_id_switches": 0}
    span = range(min(known), max(known) + 1)
    kf = np.array(sorted(known))
    kx = np.array([known[f][0] for f in kf]); ky = np.array([known[f][1] for f in kf])

    seq = []   # (frame, 찾은 track_id 또는 None, 정답 위치가 확실한지)
    for fid in span:
        gx, gy = (known[fid] if fid in known else (np.interp(fid, kf, kx), np.interp(fid, kf, ky)))
        best, bd = None, 1e9
        for p in run_by_frame.get(fid, []):
            d = np.hypot(p["feet"][0] - gx, p["feet"][1] - gy)
            if d < bd:
                best, bd = p, d
        tid = best["track_id"] if best is not None and bd <= SCORER_MATCH_PX else None
        seq.append((fid, tid, fid in known))

    sure = [t for _, t, k in seq if k and t is not None]
    switches = sum(1 for a, b in zip(sure, sure[1:]) if a != b)
    covered = sum(1 for _, t, _ in seq if t is not None)

    if verbose:   # 어느 ID가 어느 구간을 맡았는지 (v3 노트북의 SEGMENTS와 같은 형태) → 영상으로 확인할 때 사용
        runs, cur = [], None
        for fid, t, _ in seq:
            if cur and cur[0] == t and cur[2] == fid - 1:
                cur[2] = fid
            else:
                cur = [t, fid, fid]; runs.append(cur)
        desc = ", ".join(f"{'없음' if t is None else '#' + str(t)}({a}~{b})" for t, a, b in runs)
        print(f"  [{stem}] 득점 선수 경로: {desc}")

    return {
        "scorer_frames": len(seq),
        "scorer_coverage": round(covered / len(seq), 3),
        "scorer_ids": len(set(sure)),
        "scorer_id_switches": switches,
    }


def segment_stats(stem, run_json, cfg):
    """플레이 구간 안의 고유 ID 수, Unknown 비율 (라벨링 없이 측정)"""
    by_frame, _ = load_tracking(run_json)
    a, b = cfg["play_range"]
    ids, types = set(), Counter()
    for fid in range(a, b + 1):
        for p in by_frame.get(fid, []):
            ids.add(p["track_id"]); types[p["type"]] += 1
    total = sum(types.values())
    return {"segment_unique_ids": len(ids),
            "segment_unknown_ratio": round(types["Unknown"] / total, 3) if total else 0.0}


RESULTS_CSV = os.path.join(EVAL_DIR, "eval_results.csv")


def evaluate_run(run_name, run_json_by_stem, note=""):
    """
    한 번의 실험 결과를 채점해서 eval_results.csv에 한 줄 추가합니다.
    run_json_by_stem: {stem: 그 실험이 만든 tracking_final.json 경로}
    """
    det, conf_mat, per_frame = detection_and_team_metrics(run_json_by_stem)
    row = {"run": run_name, "note": note, **det}
    for stem, jpath in run_json_by_stem.items():
        cfg = EVAL_CLIPS[stem]
        k = cfg["clip_key"]
        for key, val in {**scorer_track_metrics(stem, jpath, cfg), **segment_stats(stem, jpath, cfg)}.items():
            row[f"{key}_{k}"] = val
    df_new = pd.DataFrame([row])
    if os.path.exists(RESULTS_CSV):
        old = pd.read_csv(RESULTS_CSV)
        old = old[old["run"] != run_name]                      # 같은 이름으로 다시 돌리면 덮어쓰기
        df_new = pd.concat([old, df_new], ignore_index=True)
    df_new.to_csv(RESULTS_CSV, index=False, encoding="utf-8-sig")
    return row, conf_mat, per_frame


def show_confusion(conf_mat):
    rows = EVAL_TEAM + (["Unknown"] if any(g == "Unknown" for g, _ in conf_mat) else [])
    cols = CLASS_NAMES
    return pd.DataFrame([[conf_mat.get((r, c), 0) for c in cols] for r in rows],
                        index=[f"정답:{r}" for r in rows], columns=[f"예측:{c}" for c in cols])

print("✅ 평가 함수 준비 완료")

In [ ]:
# 틀린 곳을 눈으로 확인: 초록=맞게 찾음(TP), 빨강=놓친 정답(FN), 파랑=잘못 잡음(FP)
def show_errors(name, run_json_by_stem):
    stem = name.split("_f")[0]
    fid = int(name.split("_f")[-1])
    by_frame, poly = load_tracking(run_json_by_stem[stem])
    preds = by_frame.get(fid, [])
    g = GT[name]
    pairs, ug, up = match_by_center([b for _, b in g["boxes"]], [p["bbox"] for p in preds])
    img = cv2.imread(os.path.join(IMG_DIR, name + ".jpg"))
    cv2.polylines(img, [poly.reshape(-1, 1, 2)], True, (0, 255, 0), 1)
    for gi, pi in pairs:
        b = preds[pi]["bbox"]; cv2.rectangle(img, (int(b[0]), int(b[1])), (int(b[2]), int(b[3])), (0, 200, 0), 2)
    for gi in ug:
        b = g["boxes"][gi][1]; cv2.rectangle(img, (int(b[0]), int(b[1])), (int(b[2]), int(b[3])), (0, 0, 255), 2)
    for pi in up:
        b = preds[pi]["bbox"]; cv2.rectangle(img, (int(b[0]), int(b[1])), (int(b[2]), int(b[3])), (255, 0, 0), 2)
    plt.figure(figsize=(16, 9)); plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(f"{name}  TP={len(pairs)}  FN={len(ug)}  FP={len(up)}"); plt.axis("off"); plt.show()

## ByteTrack
v3는 Ultralytics 안의 ByteTrack을 썼어요. 여기서는 저장해둔 탐지 결과에 설정만 바꿔 여러 번 돌리려고,
같은 규칙을 numpy로 옮긴 버전을 씁니다. 셀 끝의 자체 점검이 통과하는지 확인하세요.

In [ ]:
# ===== ByteTrack (논문 · Ultralytics 구현과 같은 규칙을 그대로 옮긴 버전) =====
# 저장해둔 탐지 결과에 바로 돌릴 수 있도록 numpy만으로 작성했습니다.
#   1) 칼만 필터로 기존 트랙의 이번 위치 예측
#   2) 높은 점수 박스와 IoU 짝짓기 (헝가리안)
#   3) 짝 못 찾은 트랙은 낮은 점수 박스와 한 번 더 짝짓기  ← ByteTrack의 핵심
#   4) 남은 높은 점수 박스 중 new_track_thresh 이상만 새 트랙으로 등록
#   5) track_buffer 프레임 동안 못 찾은 트랙은 삭제

class KalmanXYAH:
    """상태: [cx, cy, 종횡비 a, 높이 h, 각 속도 4개]. 등속 모델."""
    def __init__(self):
        self.F = np.eye(8)
        for i in range(4):
            self.F[i, 4 + i] = 1.0
        self.H = np.eye(4, 8)
        self.wp, self.wv = 1.0 / 20, 1.0 / 160

    def initiate(self, z):
        h = z[3]
        mean = np.r_[z, np.zeros(4)]
        std = [2 * self.wp * h, 2 * self.wp * h, 1e-2, 2 * self.wp * h,
               10 * self.wv * h, 10 * self.wv * h, 1e-5, 10 * self.wv * h]
        return mean, np.diag(np.square(std))

    def predict(self, mean, cov):
        h = mean[3]
        q = [self.wp * h, self.wp * h, 1e-2, self.wp * h, self.wv * h, self.wv * h, 1e-5, self.wv * h]
        return self.F @ mean, self.F @ cov @ self.F.T + np.diag(np.square(q))

    def update(self, mean, cov, z):
        h = mean[3]
        R = np.diag(np.square([self.wp * h, self.wp * h, 1e-1, self.wp * h]))
        S = self.H @ cov @ self.H.T + R
        K = np.linalg.solve(S, (cov @ self.H.T).T).T
        mean = mean + K @ (z - self.H @ mean)
        cov = cov - K @ S @ K.T
        return mean, cov


def xyxy_to_xyah(b):
    w, h = b[2] - b[0], b[3] - b[1]
    return np.array([(b[0] + b[2]) / 2, (b[1] + b[3]) / 2, w / max(h, 1e-6), h], float)


def xyah_to_xyxy(m):
    cx, cy, a, h = m[:4]
    w = a * h
    return np.array([cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2], float)


def iou_matrix_np(a, b):
    a = np.asarray(a, float).reshape(-1, 4); b = np.asarray(b, float).reshape(-1, 4)
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)))
    lt = np.maximum(a[:, None, :2], b[None, :, :2]); rb = np.minimum(a[:, None, 2:], b[None, :, 2:])
    wh = np.clip(rb - lt, 0, None); inter = wh[..., 0] * wh[..., 1]
    area_a = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1]); area_b = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    return inter / (area_a[:, None] + area_b[None, :] - inter + 1e-9)


def linear_assign(cost, thresh):
    if cost.size == 0:
        return [], list(range(cost.shape[0])), list(range(cost.shape[1]))
    c = np.where(cost <= thresh, cost, 1e6)
    r, k = linear_sum_assignment(c)
    m = [(i, j) for i, j in zip(r, k) if c[i, j] <= thresh]
    mi = {i for i, _ in m}; mj = {j for _, j in m}
    return m, [i for i in range(cost.shape[0]) if i not in mi], [j for j in range(cost.shape[1]) if j not in mj]


class Track:
    _next_id = 0
    TRACKED, LOST, REMOVED = 0, 1, 2

    def __init__(self, box, score, det_idx, kf):
        self.kf, self.box, self.score, self.det_idx = kf, np.asarray(box, float), score, det_idx
        self.mean = self.cov = None
        self.state, self.activated, self.track_id = Track.TRACKED, False, -1
        self.start_frame = self.end_frame = 0

    def predicted_box(self):
        return xyah_to_xyxy(self.mean) if self.mean is not None else self.box

    def predict(self):
        m = self.mean.copy()
        if self.state != Track.TRACKED:
            m[7] = 0.0                          # 놓친 트랙은 높이 변화 속도 0
        self.mean, self.cov = self.kf.predict(m, self.cov)

    def activate(self, frame_id):
        Track._next_id += 1
        self.track_id = Track._next_id
        self.mean, self.cov = self.kf.initiate(xyxy_to_xyah(self.box))
        self.state = Track.TRACKED
        self.activated = frame_id == 1
        self.start_frame = self.end_frame = frame_id

    def update(self, det, frame_id):
        self.mean, self.cov = self.kf.update(self.mean, self.cov, xyxy_to_xyah(det.box))
        self.box, self.score, self.det_idx = det.box, det.score, det.det_idx
        self.state, self.activated, self.end_frame = Track.TRACKED, True, frame_id


class ByteTracker:
    def __init__(self, track_high_thresh=0.5, track_low_thresh=0.1, new_track_thresh=0.6,
                 track_buffer=30, match_thresh=0.8, fuse_score=True, frame_rate=30):
        self.hi, self.lo, self.new = track_high_thresh, track_low_thresh, new_track_thresh
        self.max_lost = int(frame_rate / 30.0 * track_buffer)
        self.match, self.fuse = match_thresh, fuse_score
        self.kf = KalmanXYAH()
        self.tracked, self.lost = [], []
        self.frame_id = 0
        Track._next_id = 0

    def _dist(self, tracks, dets, fuse):
        d = 1 - iou_matrix_np([t.predicted_box() for t in tracks], [x.box for x in dets])
        if fuse and len(dets):
            d = 1 - (1 - d) * np.array([x.score for x in dets])[None, :]
        return d

    def update(self, boxes, scores):
        """boxes (N,4) xyxy, scores (N,) → [(track_id, det_idx), ...] 이번 프레임에 확정된 트랙"""
        self.frame_id += 1
        f = self.frame_id
        boxes = np.asarray(boxes, float).reshape(-1, 4); scores = np.asarray(scores, float).reshape(-1)
        high = [Track(boxes[i], scores[i], i, self.kf) for i in np.where(scores >= self.hi)[0]]
        low = [Track(boxes[i], scores[i], i, self.kf) for i in np.where((scores > self.lo) & (scores < self.hi))[0]]

        unconfirmed = [t for t in self.tracked if not t.activated]
        confirmed = [t for t in self.tracked if t.activated]
        pool = confirmed + [t for t in self.lost if t not in confirmed]
        for t in pool:
            t.predict()

        refound, activated_now = [], []
        # 1차: 높은 점수 박스
        m, u_t, u_d = linear_assign(self._dist(pool, high, self.fuse), self.match)
        for i, j in m:
            t = pool[i]
            if t.state != Track.TRACKED:
                refound.append(t)
            t.update(high[j], f); activated_now.append(t)
        # 2차: 남은 '추적 중' 트랙 × 낮은 점수 박스
        r_tracked = [pool[i] for i in u_t if pool[i].state == Track.TRACKED]
        m2, u_t2, _ = linear_assign(self._dist(r_tracked, low, False), 0.5)
        for i, j in m2:
            r_tracked[i].update(low[j], f); activated_now.append(r_tracked[i])
        for i in u_t2:
            r_tracked[i].state = Track.LOST
        # 확정 안 된(1프레임짜리) 트랙 × 남은 높은 점수 박스
        rest_high = [high[j] for j in u_d]
        m3, u_un, u_d3 = linear_assign(self._dist(unconfirmed, rest_high, self.fuse), 0.7)
        for i, j in m3:
            unconfirmed[i].update(rest_high[j], f); activated_now.append(unconfirmed[i])
        for i in u_un:
            unconfirmed[i].state = Track.REMOVED
        # 새 트랙
        new_tracks = []
        for j in u_d3:
            d = rest_high[j]
            if d.score >= self.new:
                d.activate(f); new_tracks.append(d)
        # 오래 못 찾은 트랙 삭제
        for t in pool:
            if t.state == Track.LOST and f - t.end_frame > self.max_lost:
                t.state = Track.REMOVED

        everyone = {id(t): t for t in self.tracked + self.lost + new_tracks}
        self.tracked = [t for t in everyone.values() if t.state == Track.TRACKED]
        self.lost = [t for t in everyone.values() if t.state == Track.LOST]
        return [(t.track_id, t.det_idx) for t in self.tracked if t.activated and t.end_frame == f]


def _bytetrack_selftest():
    """작은 합성 장면으로 동작 확인: 점수가 잠깐 낮아지거나 몇 프레임 사라져도 ID 유지, 약한 새 박스는 새 ID 안 받음"""
    tr = ByteTracker(track_high_thresh=0.5, track_low_thresh=0.1, new_track_thresh=0.6, track_buffer=30)
    ids = defaultdict(set)
    for f in range(1, 61):
        boxes, scores = [], []
        x = 100 + 3 * f                                     # 선수 A: 오른쪽으로 이동
        if not (30 <= f < 36):                              # 30~35프레임 동안 사라짐(가려짐)
            boxes.append([x, 100, x + 12, 130]); scores.append(0.3 if 15 <= f < 20 else 0.8)   # 15~19 낮은 점수
        boxes.append([300, 200 + f, 312, 230 + f]); scores.append(0.9)                      # 선수 B
        if f == 40:
            boxes.append([500, 50, 512, 80]); scores.append(0.4)                            # 약한 오탐 1번
        for tid, di in tr.update(boxes, scores):
            ids["A" if boxes[di][0] < 290 else ("B" if boxes[di][0] < 400 else "X")].add(tid)
    assert len(ids["A"]) == 1 and len(ids["B"]) == 1 and "X" not in ids, dict(ids)
    print("✅ ByteTrack 자체 점검 통과:", {k: sorted(v) for k, v in ids.items()})

_bytetrack_selftest()

---
# 3b. 팀 분류기 재학습 (PyTorch · ResNet18 전이 학습)

| 순서 | 내용 | 누가 / 시간 |
|---|---|---|
| ① | 2a·2b의 탐지 캐시 + 튜닝된 추적기 불러오기 | 자동, 1분 |
| ② | 튜닝 클립(0143·0157)에서 트랙 뽑기 + 선수 사진 잘라두기 | 자동, 2~3분 |
| ③ | **트랙마다 사진을 보고 [홍익] [국민] [심판] [제외] 버튼 누르기** | 직접, 10~20분 |
| ④ | 트랙 단위로 학습/검증 나누기 | 자동 |
| ⑤ | ResNet18 학습 (ImageNet 사전학습 → 우리 데이터로 미세조정) | 자동, 2~5분 |
| ⑥ | 평가 클립에 적용 → v5 채점 → v4와 비교 | 자동, 3~5분 |

- 학습 데이터는 **평가 클립이 아닌 튜닝 클립**에서만 만들어요. 평가 정답(20장)은 ⑥ 채점에만 씁니다.
- 학습/검증은 **트랙 단위**로 나눠요. 같은 선수의 비슷한 사진이 학습과 검증에 동시에 들어가면 검증 점수가 부풀려지기 때문이에요.

**런타임: T4 GPU**

In [ ]:
import time, pickle, random
import torch, torch.nn as nn
import torchvision
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("PyTorch", torch.__version__, "| torchvision", torchvision.__version__, "| 장치:", DEVICE)

SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

CACHE_DIR = os.path.join(EVAL_DIR, "cache")
TUNE_CLIPS = ["DJI_20250913133530_0143_D", "DJI_20250913135111_0157_D"]
TEAM_CLASSES = ["Hongik", "Kookmin", "Referee"]
LABELS_PATH = os.path.join(EVAL_DIR, "team_track_labels.json")      # 라벨링 결과 (중간 저장 → 이어서 하기 가능)
MODEL_PATH = os.path.join(EVAL_DIR, "team_cnn_resnet18.pt")

_tcfg = json.load(open(os.path.join(EVAL_DIR, "tracker_config_v4.json"), encoding="utf-8"))
TRACK_PARAMS = _tcfg["params"]
print("추적기 설정 (2a 튜닝):", TRACK_PARAMS)


def load_pkl(path):
    assert os.path.exists(path), f"캐시가 없습니다: {path}\n→ 2a(튜닝 클립)·2b(평가 클립) 노트북을 먼저 실행하세요."
    with open(path, "rb") as f:
        return pickle.load(f)

TUNE_CACHE = {s: load_pkl(os.path.join(CACHE_DIR, f"{s}_tune_y26x1920_conf0.05.pkl")) for s in TUNE_CLIPS}
EVAL_CACHE = {s: load_pkl(os.path.join(CACHE_DIR, f"{s}_y26x1920_conf0.05_full.pkl")) for s in EVAL_CLIPS}
for s, c in {**TUNE_CACHE, **EVAL_CACHE}.items():
    print(f"  {s}: 프레임 {c['start']}~{c['end']}")


def run_tracker_on(cache, params=TRACK_PARAMS):
    tr = ByteTracker(**params, fuse_score=True, frame_rate=30)
    out = {}
    for fid in range(cache["start"], cache["end"] + 1):
        fr = cache["frames"].get(fid, {"boxes": np.zeros((0, 4)), "scores": np.zeros(0)})
        out[fid] = tr.update(fr["boxes"], fr["scores"])
    return out


def pad_crop(img, box, pad=0.15):
    """박스를 살짝 넓혀서 자름 (선수가 작아서 테두리 정보가 도움이 됨)"""
    H, W = img.shape[:2]
    x1, y1, x2, y2 = box
    w, h = x2 - x1, y2 - y1
    x1, x2 = int(max(0, x1 - pad * w)), int(min(W, x2 + pad * w))
    y1, y2 = int(max(0, y1 - pad * h)), int(min(H, y2 + pad * h))
    return img[y1:y2, x1:x2].copy() if x2 > x1 and y2 > y1 else None


def collect_crops(stem, wanted):
    """wanted: {frame_id: [(key, box), ...]} → {key: crop}. 영상을 처음부터 순서대로 읽어서 자름."""
    local = os.path.join("/content/_scratch_video", os.path.basename(video_path_of(stem)))
    os.makedirs(os.path.dirname(local), exist_ok=True)
    if not os.path.exists(local):
        shutil.copy(video_path_of(stem), local)
    cap = cv2.VideoCapture(local)
    out, fid, last = {}, 0, max(wanted) if wanted else 0
    while fid < last:
        ok, img = cap.read()
        if not ok:
            break
        fid += 1
        for key, box in wanted.get(fid, []):
            cr = pad_crop(img, box)
            if cr is not None and cr.size:
                out[key] = cr
    cap.release()
    return out

In [ ]:
# ② 튜닝 클립에서 트랙 뽑기 + 사진 자르기
MIN_TRACK_LEN = 20          # 20프레임(0.7초) 미만 트랙은 라벨링·학습에서 제외 (잡음이 많음)
SAMPLE_EVERY = 3            # 학습용 사진은 3프레임마다 한 장 (연속 프레임은 거의 똑같아서)
MONTAGE_N = 8               # 라벨링 화면에 보여줄 사진 수 (트랙 전체에서 고르게)

TRACKS = {}                 # (stem, tid) → [(fid, box), ...]
for stem in TUNE_CLIPS:
    c = TUNE_CACHE[stem]
    for fid, lst in run_tracker_on(c).items():
        for tid, di in lst:
            TRACKS.setdefault((stem, int(tid)), []).append((fid, c["frames"][fid]["boxes"][di]))
TRACKS = {k: v for k, v in TRACKS.items() if len(v) >= MIN_TRACK_LEN}
print(f"라벨링할 트랙: {len(TRACKS)}개 (길이 {MIN_TRACK_LEN}프레임 이상)")

CROPS = {}                  # (stem, tid, fid) → 사진
for stem in TUNE_CLIPS:
    wanted = defaultdict(list)
    for (s, tid), seq in TRACKS.items():
        if s != stem:
            continue
        idx = set(range(0, len(seq), SAMPLE_EVERY)) | set(np.linspace(0, len(seq) - 1, MONTAGE_N).round().astype(int).tolist())
        for i in sorted(idx):
            fid, box = seq[i]
            wanted[fid].append(((stem, tid, fid), box))
    t0 = time.time()
    CROPS.update(collect_crops(stem, wanted))
    print(f"  {stem}: 사진 {sum(1 for k in CROPS if k[0] == stem):,}장 ({time.time() - t0:.0f}s)")


def montage(stem, tid, n=MONTAGE_N, h=96):
    seq = TRACKS[(stem, tid)]
    picks = np.linspace(0, len(seq) - 1, n).round().astype(int)
    tiles = []
    for i in picks:
        cr = CROPS.get((stem, tid, seq[i][0]))
        if cr is None:
            continue
        w = max(1, int(cr.shape[1] * h / cr.shape[0]))
        tiles.append(cv2.resize(cr, (w, h), interpolation=cv2.INTER_NEAREST))
        tiles.append(np.full((h, 4, 3), 255, np.uint8))
    return np.hstack(tiles) if tiles else np.zeros((h, h, 3), np.uint8)

## ③ 트랙 라벨링

한 줄에 **같은 트랙(같은 ID)의 사진 8장**이 시간 순서대로 나와요. 보고 버튼을 누르면 다음 트랙으로 넘어가요.

| 버튼 | 언제 |
|---|---|
| **홍익** / **국민** / **심판** | 8장 대부분이 그 사람일 때 |
| **제외** | 사람이 아니거나, 중간에 다른 사람으로 바뀌었거나, 너무 흐려서 모를 때 |
| **↩ 이전** | 방금 누른 것을 고치고 싶을 때 |

- 누를 때마다 Drive에 저장돼요. 중간에 끊겨도 이 셀을 다시 실행하면 **이어서** 할 수 있어요.
- 사이드라인 코치·대기 선수가 섞여 나오면 **유니폼 기준**으로 고르거나, 경기 인원이 아니면 **제외**하세요.
- 긴 트랙부터 나와요. 시간이 부족하면 **도중에 멈추고 ④로 넘어가도 돼요** (라벨한 트랙만 학습에 써요). 심판 트랙은 꼭 몇 개 포함되게 해주세요.

In [ ]:
import ipywidgets as widgets
from IPython.display import display as _display, clear_output

LABELS = json.load(open(LABELS_PATH, encoding="utf-8")) if os.path.exists(LABELS_PATH) else {}
ORDER = sorted(TRACKS, key=lambda k: -len(TRACKS[k]))            # 긴 트랙부터
key_str = lambda k: f"{k[0]}|{k[1]}"
HISTORY = []


def save_labels():
    with open(LABELS_PATH, "w", encoding="utf-8") as f:
        json.dump(LABELS, f, ensure_ascii=False)


def next_key():
    for k in ORDER:
        if key_str(k) not in LABELS:
            return k
    return None


out = widgets.Output()
status = widgets.HTML()
btns = {name: widgets.Button(description=desc, layout=widgets.Layout(width="110px"), button_style=style)
        for name, desc, style in [("Hongik", "홍익", "info"), ("Kookmin", "국민", "danger"),
                                  ("Referee", "심판", "warning"), ("X", "제외", ""), ("undo", "↩ 이전", "")]}


def render():
    k = next_key()
    done = sum(1 for kk in ORDER if key_str(kk) in LABELS)
    cnt = Counter(v for v in LABELS.values())
    status.value = (f"<b>{done} / {len(ORDER)}</b> 완료 &nbsp; (홍익 {cnt['Hongik']} · 국민 {cnt['Kookmin']} · "
                    f"심판 {cnt['Referee']} · 제외 {cnt['X']})")
    with out:
        clear_output(wait=True)
        if k is None:
            print("✅ 라벨링 완료! 다음 셀(④)로 넘어가세요.")
            return
        m = montage(*k)
        plt.figure(figsize=(min(18, 1.6 * MONTAGE_N), 2.2))
        plt.imshow(cv2.cvtColor(m, cv2.COLOR_BGR2RGB)); plt.axis("off")
        plt.title(f"{k[0][-6:-2]} 클립 · 트랙 #{k[1]} · 길이 {len(TRACKS[k])}프레임"); plt.show()


def on_click(name):
    def _f(_):
        if name == "undo":
            if HISTORY:
                LABELS.pop(HISTORY.pop(), None)
        else:
            k = next_key()
            if k is not None:
                LABELS[key_str(k)] = name; HISTORY.append(key_str(k))
        save_labels(); render()
    return _f


for name, b in btns.items():
    b.on_click(on_click(name))
_display(widgets.VBox([status, widgets.HBox(list(btns.values())), out]))
render()

> 버튼이 보이지 않거나 눌러도 반응이 없으면, 이 셀 위에 새 셀을 만들어
> `from google.colab import output; output.enable_custom_widget_manager()` 를 실행한 뒤 다시 실행하세요.

In [ ]:
# ④ 학습 / 검증 나누기 (트랙 단위, 클래스별로 20%를 검증에)
LABELS = json.load(open(LABELS_PATH, encoding="utf-8"))
by_cls = defaultdict(list)
for ks, lab in LABELS.items():
    if lab in TEAM_CLASSES:
        stem, tid = ks.split("|")
        by_cls[lab].append((stem, int(tid)))
rng_split = random.Random(SEED)
TRAIN_TRACKS, VAL_TRACKS = [], []
for lab, ks in by_cls.items():
    ks = sorted(ks); rng_split.shuffle(ks)
    n_val = max(1, round(0.2 * len(ks))) if len(ks) >= 2 else 0
    VAL_TRACKS += [(k, lab) for k in ks[:n_val]]
    TRAIN_TRACKS += [(k, lab) for k in ks[n_val:]]


def samples_of(track_list):
    out = []
    for (stem, tid), lab in track_list:
        seq = TRACKS[(stem, tid)]
        for i in range(0, len(seq), SAMPLE_EVERY):
            cr = CROPS.get((stem, tid, seq[i][0]))
            if cr is not None:
                out.append((cr, TEAM_CLASSES.index(lab), (stem, tid)))
    return out

TRAIN_S, VAL_S = samples_of(TRAIN_TRACKS), samples_of(VAL_TRACKS)
summary = pd.DataFrame({
    "학습 트랙": Counter(l for _, l in TRAIN_TRACKS), "검증 트랙": Counter(l for _, l in VAL_TRACKS),
    "학습 사진": Counter(TEAM_CLASSES[y] for _, y, _ in TRAIN_S), "검증 사진": Counter(TEAM_CLASSES[y] for _, y, _ in VAL_S),
}).reindex(TEAM_CLASSES).fillna(0).astype(int)
display(summary)
assert len(TRAIN_S) and len(VAL_S), "학습/검증 사진이 없습니다. ③ 라벨링을 먼저 끝내세요."
if summary.loc["Referee", "학습 트랙"] < 2:
    print("⚠️ 심판 트랙이 2개 미만이에요. 심판 분류는 기대하기 어려워요. 가능하면 심판 트랙을 더 라벨링하세요.")

In [ ]:
# ⑤ ResNet18 전이 학습
IMG_H, IMG_W = 128, 64       # 선수는 세로로 긴 모양이라 2:1
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]   # ImageNet 정규화 (사전학습 모델과 같은 기준)
EPOCHS, BATCH, LR = 8, 64, 3e-4

train_tf = transforms.Compose([
    transforms.ToPILImage(), transforms.Resize((IMG_H, IMG_W)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.25, contrast=0.2),   # 햇빛·그늘 변화 대비. 색조(hue)는 팀 구분 단서라 건드리지 않음
    transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
test_tf = transforms.Compose([
    transforms.ToPILImage(), transforms.Resize((IMG_H, IMG_W)),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD)])


class CropDataset(Dataset):
    def __init__(self, samples, tf):
        self.s, self.tf = samples, tf
    def __len__(self):
        return len(self.s)
    def __getitem__(self, i):
        cr, y, _ = self.s[i]
        return self.tf(cv2.cvtColor(cr, cv2.COLOR_BGR2RGB)), y


def build_model():
    m = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.IMAGENET1K_V1)
    m.fc = nn.Linear(m.fc.in_features, len(TEAM_CLASSES))       # 마지막 층만 3클래스로 교체
    return m.to(DEVICE)


@torch.no_grad()
def predict_probs(model, crops, bs=256):
    """사진 리스트 → (N, 3) 확률"""
    model.eval()
    out = []
    for i in range(0, len(crops), bs):
        x = torch.stack([test_tf(cv2.cvtColor(c, cv2.COLOR_BGR2RGB)) for c in crops[i:i + bs]]).to(DEVICE)
        out.append(torch.softmax(model(x), 1).cpu().numpy())
    return np.concatenate(out) if out else np.zeros((0, len(TEAM_CLASSES)))


def per_class_recall(y_true, y_pred):
    return {c: round(float(np.mean(y_pred[y_true == i] == i)), 3) if np.any(y_true == i) else None
            for i, c in enumerate(TEAM_CLASSES)}


# 클래스 불균형(심판이 적음) → 적은 클래스의 오답에 더 큰 벌점
counts = np.bincount([y for _, y, _ in TRAIN_S], minlength=len(TEAM_CLASSES)).astype(float)
class_w = torch.tensor(counts.sum() / (len(counts) * np.maximum(counts, 1)), dtype=torch.float32, device=DEVICE)
print("클래스 가중치:", dict(zip(TEAM_CLASSES, class_w.cpu().numpy().round(2))))

model = build_model()
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
loss_fn = nn.CrossEntropyLoss(weight=class_w)
train_dl = DataLoader(CropDataset(TRAIN_S, train_tf), batch_size=BATCH, shuffle=True, num_workers=2)
y_val = np.array([y for _, y, _ in VAL_S])

best, history = (-1, None), []
for ep in range(1, EPOCHS + 1):
    model.train(); t0 = time.time(); tot = 0.0
    for x, y in train_dl:
        x, y = x.to(DEVICE), y.to(DEVICE)
        opt.zero_grad(); loss = loss_fn(model(x), y); loss.backward(); opt.step()
        tot += loss.item() * len(y)
    sched.step()
    pv = predict_probs(model, [c for c, _, _ in VAL_S]).argmax(1)
    rec = per_class_recall(y_val, pv)
    macro = float(np.mean([v for v in rec.values() if v is not None]))
    history.append({"epoch": ep, "train_loss": round(tot / len(TRAIN_S), 4), "val_acc": round(float(np.mean(pv == y_val)), 3),
                    "val_macro_recall": round(macro, 3), **{f"val_{k}": v for k, v in rec.items()}})
    print(history[-1], f"({time.time() - t0:.0f}s)")
    if macro > best[0]:
        best = (macro, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()})

model.load_state_dict(best[1])
torch.save({"state_dict": best[1], "classes": TEAM_CLASSES, "img_hw": (IMG_H, IMG_W), "history": history}, MODEL_PATH)
print(f"✅ 검증 macro recall이 가장 좋은 모델 저장 ({best[0]:.3f}) → {MODEL_PATH}")

# 검증 결과 (사진 단위 / 트랙 단위)
pv_prob = predict_probs(model, [c for c, _, _ in VAL_S]); pv = pv_prob.argmax(1)
cm = pd.crosstab(pd.Series([TEAM_CLASSES[i] for i in y_val], name="정답"), pd.Series([TEAM_CLASSES[i] for i in pv], name="예측"))
print("\n=== 검증 혼동행렬 (사진 단위) ==="); display(cm)
trk_prob = defaultdict(list)
for (cr, y, k), p in zip(VAL_S, pv_prob):
    trk_prob[(k, y)].append(p)
trk_ok = [int(np.mean(ps, 0).argmax() == y) for (k, y), ps in trk_prob.items()]
print(f"검증 트랙 정확도 (트랙 평균 확률): {np.mean(trk_ok):.3f}  ({sum(trk_ok)}/{len(trk_ok)} 트랙)")
plt.figure(figsize=(8, 3.5)); hdf = pd.DataFrame(history)
plt.plot(hdf["epoch"], hdf["train_loss"], "o-", label="학습 손실"); plt.plot(hdf["epoch"], hdf["val_macro_recall"], "s-", label="검증 macro recall")
plt.xlabel("epoch"); plt.legend(); plt.grid(alpha=0.3); plt.title("학습 곡선"); plt.show()

In [ ]:
# ⑥ 평가 클립에 적용 → v5 JSON → 채점
MAX_REFEREE_PER_FRAME, MAX_PLAYERS_PER_TEAM_PER_FRAME = 3, 11


def classify_eval_clip(stem):
    c = EVAL_CACHE[stem]
    tracks = run_tracker_on(c)
    wanted = defaultdict(list)
    for fid, lst in tracks.items():
        for tid, di in lst:
            wanted[fid].append(((fid, int(tid)), c["frames"][fid]["boxes"][di]))
    t0 = time.time()
    crops = collect_crops(stem, wanted)
    keys = list(crops)
    probs = dict(zip(keys, predict_probs(model, [crops[k] for k in keys])))
    print(f"  {stem}: 사진 {len(keys):,}장 분류 ({time.time() - t0:.0f}s)")
    return tracks, probs


def build_v5_json(stem, tracks, probs, run_dir, ref_cap, team_cap, note):
    c = EVAL_CACHE[stem]
    # 트랙 라벨 = 트랙 전체 확률의 평균이 가장 높은 클래스 (다수결보다 흔들림이 적음)
    acc = defaultdict(list)
    for k, p in probs.items():
        acc[k[1]].append(p)
    track_p = {tid: np.mean(ps, 0) for tid, ps in acc.items()}
    frames_out = []
    for fid in sorted(tracks):
        items = []
        for tid, di in tracks[fid]:
            tid = int(tid); p = track_p.get(tid)
            lab = TEAM_CLASSES[int(np.argmax(p))] if p is not None else "Unknown"
            items.append({"tid": tid, "label": lab, "p": p, "bb": c["frames"][fid]["boxes"][di], "sc": float(c["frames"][fid]["scores"][di])})
        pr = lambda it, l: float(it["p"][TEAM_CLASSES.index(l)]) if it["p"] is not None else 0.0
        if ref_cap:
            refs = sorted([it for it in items if it["label"] == "Referee"], key=lambda it: -pr(it, "Referee"))
            for it in refs[MAX_REFEREE_PER_FRAME:]:
                it["label"] = max(["Hongik", "Kookmin"], key=lambda l: pr(it, l))
        if team_cap:
            for team in ("Hongik", "Kookmin"):
                mem = sorted([it for it in items if it["label"] == team], key=lambda it: -pr(it, team))
                for it in mem[MAX_PLAYERS_PER_TEAM_PER_FRAME:]:
                    it["label"] = "Unknown"
        players = []
        for it in items:
            x1, y1, x2, y2 = [int(round(v)) for v in it["bb"]]
            players.append({"track_id": it["tid"], "type": it["label"], "bbox": [x1, y1, x2, y2],
                            "feet": [(x1 + x2) // 2, y2], "conf": it["sc"]})
        frames_out.append({"frame_id": int(fid), "players": players})
    _, poly = load_tracking(os.path.join(EVAL_DIR, "runs", "step1a_roi", f"{stem}_tracking_final.json"))
    data = {"video_name": os.path.basename(video_path_of(stem)), "fps": 30, "field_polygon": poly.tolist(),
            "frames": frames_out, "notes": [note]}
    os.makedirs(run_dir, exist_ok=True)
    p = os.path.join(run_dir, f"{stem}_tracking_final.json")
    with open(p, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False)
    return p


EVAL_TP = {stem: classify_eval_clip(stem) for stem in EVAL_CLIPS}
V5_VARIANTS = {"v5_cnn": dict(ref_cap=False, team_cap=False), "v5_cnn_rules": dict(ref_cap=True, team_cap=True)}
CONF_V5 = {}
for run_name, flags in V5_VARIANTS.items():
    note = f"v4 탐지·추적 + ResNet18 팀 분류기(튜닝 클립 트랙 라벨) + 인원 규칙 {'O' if flags['team_cap'] else 'X'}"
    run = {stem: build_v5_json(stem, *EVAL_TP[stem], os.path.join(EVAL_DIR, "runs", run_name), note=note, **flags) for stem in EVAL_CLIPS}
    _, CONF_V5[run_name], _ = evaluate_run(run_name, run, note=note)

In [ ]:
# 비교: v4(RF+규칙) · v4_rule_none(RF) · v5_cnn · v5_cnn_rules
res = pd.read_csv(RESULTS_CSV).set_index("run")
order = [r for r in ["v4", "v4_rule_none", "v5_cnn", "v5_cnn_rules"] if r in res.index]
display(res.loc[order, ["team_acc", "referee_recall", "segment_unknown_ratio_34", "segment_unknown_ratio_44", "det_f1"]].T)


def team_breakdown(conf):
    out = {}
    for g in TEAM_CLASSES:
        tot = sum(v for (gg, _), v in conf.items() if gg == g)
        if tot:
            out[f"{g} 정답률"] = round(conf.get((g, g), 0) / tot, 3)
            out[f"{g}→Unknown"] = round(conf.get((g, "Unknown"), 0) / tot, 3)
    return out

display(pd.DataFrame({k: team_breakdown(v) for k, v in CONF_V5.items()}))
print("\n=== v5_cnn 혼동행렬 ===")
display(show_confusion(CONF_V5["v5_cnn"]))

## 결과 공유
1. ④ 학습/검증 개수 표, ⑤ 학습 로그·검증 혼동행렬·트랙 정확도
2. 마지막 비교표, 팀별 분석표, `v5_cnn` 혼동행렬

**읽는 법:** `v4_rule_none`(RF 분류기, 규칙 없음)과 `v5_cnn`(새 분류기, 규칙 없음)을 비교하면 **분류기 교체 효과만** 보여요.